# Palm Images Training in Google Colab

This notebook trains a model on your **Human Palm Images** dataset from Google Drive.

**Dataset structure expected:**
- `My Drive/palm hand dataset/FEMALE/` — palm images
- `My Drive/palm hand dataset/MALE/` — palm images

**Steps:**
1. Mount Google Drive
2. Set path to your dataset
3. Load and augment images
4. Build model (transfer learning)
5. Train and save the model

---
## Step 1: Mount Google Drive

Run this cell. A popup will ask you to sign in and allow access to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2: Set your dataset path

After mounting, your Drive is at `/content/drive/MyDrive` (or `My Drive`). Adjust `DATASET_PATH` if your folder name or location is different.

In [ ]:
import os

# ----- CONNECT YOUR DRIVE FOLDER HERE -----
# 1) After Step 1 (mount Drive), your files are under /content/drive/.
# 2) DRIVE_ROOT = root of "My Drive" (Colab uses MyDrive OR "My Drive")
# 3) DATASET_FOLDER = exact name of the folder that contains FEMALE and MALE.

DRIVE_ROOT = "/content/drive/MyDrive"      # or "/content/drive/My Drive"
DATASET_FOLDER = "palm hand dataset"       # your folder name on Drive
DATASET_PATH = os.path.join(DRIVE_ROOT, DATASET_FOLDER)

if not os.path.exists(DATASET_PATH):
    other_root = "/content/drive/My Drive" if "MyDrive" in DRIVE_ROOT else "/content/drive/MyDrive"
    alt = os.path.join(other_root, DATASET_FOLDER)
    if os.path.exists(alt):
        DATASET_PATH = alt
        print(f"Using path: {DATASET_PATH}")
    else:
        print("Contents of /content/drive:", os.listdir("/content/drive"))
        raise FileNotFoundError(f"Dataset not found. Tried: {DATASET_PATH} and {alt}")

female_path = os.path.join(DATASET_PATH, "FEMALE")
male_path = os.path.join(DATASET_PATH, "MALE")

print(f"Dataset path (connected): {DATASET_PATH}")
print(f"FEMALE folder exists: {os.path.isdir(female_path)}")
print(f"MALE folder exists: {os.path.isdir(male_path)}")
if os.path.isdir(female_path):
    print(f"  FEMALE images: {len([f for f in os.listdir(female_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])}")
if os.path.isdir(male_path):
    print(f"  MALE images: {len([f for f in os.listdir(male_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])}")

---
## Step 3: Install TensorFlow (if needed) and load data

We use **Keras** with **ImageDataGenerator** to load images from folders and apply basic augmentation.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Connect to dataset: use DATASET_PATH from Step 2 (your Drive folder with FEMALE & MALE)
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

train_ds = train_datagen.flow_from_directory(
    DATASET_PATH,  # <-- dataset path from Step 2
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=SEED
)

val_ds = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=SEED
)

print("Class indices:", train_ds.class_indices)
print(f"Training samples: {train_ds.samples}")
print(f"Validation samples: {val_ds.samples}")

---
## Step 4: Build the model (transfer learning)

We use **MobileNetV2** pre-trained on ImageNet, then add a small head for your 2 classes (FEMALE/MALE).

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

model = keras.Sequential([
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

---
## Step 5: Train the model

Train for a few epochs. You can change `epochs` (e.g. 10–15) and enable GPU in Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

---
## Step 6: Save the model to Google Drive

Saves the trained model so you can download it or use it later.

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/palm_hand_model"
if not os.path.exists("/content/drive/MyDrive"):
    SAVE_PATH = "/content/drive/My Drive/palm_hand_model"

os.makedirs(SAVE_PATH, exist_ok=True)
model.save(os.path.join(SAVE_PATH, "palm_classifier.keras"))
print(f"Model saved to: {SAVE_PATH}/palm_classifier.keras")

---
## Optional: Plot accuracy and loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.show()